In [1]:
import re

# 1. RFC 822 및 RFC 3676 규약에 맞게 올바른 뉴스 포맷 문자열 작성 (생성)
formatted_news_article = """From: alice@example.com
Subject: Hello
Organization: NASA
Lines: 6

This is body text. This is the body content of the email.
I hope this information helps you.

-- 
John Doe
Senior Software Engineer
Company Inc."""


# 2. RFC 규약에 따라 Header, Body, Footer를 구분(파싱)하는 함수
def parse_rfc822_news(text):
    # A. Header와 나머지(Body + Footer) 분리: 첫 번째 빈 줄(\n\s*\n) 기준
    header_split = re.split(r'\n\s*\n', text, maxsplit=1)
    
    header = header_split[0].strip() if len(header_split) > 0 else ""
    remainder = header_split[1] if len(header_split) > 1 else ""

    # B. Body와 Footer 분리: RFC 표준 서명 구분선(\n-- \n) 기준
    footer_split = re.split(r'\n-- \n', remainder, maxsplit=1)
    
    body = footer_split[0].strip() if len(footer_split) > 0 else ""
    footer = footer_split[1].strip() if len(footer_split) > 1 else ""

    return {
        "header": header,
        "body": body,
        "footer": footer
    }


# 3. 파싱 함수 실행 및 결과 확인
parsed_result = parse_rfc822_news(formatted_news_article)

print("=== 1. HEADER (메타데이터) ===")
print(parsed_result["header"])

print("\n=== 2. BODY (순수 본문) ===")
print(parsed_result["body"])

print("\n=== 3. FOOTER (서명) ===")
print(parsed_result["footer"] if parsed_result["footer"] else "(Footer 없음)")

=== 1. HEADER (메타데이터) ===
From: alice@example.com
Subject: Hello
Organization: NASA
Lines: 6

=== 2. BODY (순수 본문) ===
This is body text. This is the body content of the email.
I hope this information helps you.

=== 3. FOOTER (서명) ===
John Doe
Senior Software Engineer
Company Inc.


In [5]:
import pandas as pd

df_loaded = pd.read_csv(r'c:\SKN35_kim\DL_WORKSPACE\data\20newsgroups_train.csv')
print(f"불러온 데이터 개수: {len(df_loaded)}")
df_loaded.head()

불러온 데이터 개수: 593


,text,target,target_name
0,From: henry@zoo.toronto.edu (Henry Spencer)\nS...,0,sci.space
1,From: henry@zoo.toronto.edu (Henry Spencer)\nS...,0,sci.space
2,From: aws@iti.org (Allen W. Sherzer)\nSubject:...,0,sci.space
3,From: pgf@srl03.cacs.usl.edu (Phil G. Fraering...,0,sci.space
4,From: Pat.Hoage@f6507.n124.z1.fidonet.org (Pat...,0,sci.space


In [6]:
# scikit-learn에서 20개 뉴스그룹 텍스트 데이터셋을 가져오는 함수 불러오기
from sklearn.datasets import fetch_20newsgroups

# 데이터 처리를 위해 pandas 라이브러리를 불러오고 pd라는 별칭으로 사용
import pandas as pd

# fetch_20newsgroups() 함수를 사용해 전체 데이터 로드
# subset='all': 학습용(train)과 평가용(test)을 합친 전체 데이터셋(~18,846개 문서)을 가져옴
news = fetch_20newsgroups(subset='all')

# 텍스트 본문(news.data)과 카테고리 라벨(news.target)을 매핑하여 Pandas DataFrame 객체 생성
# 'text': 뉴스 기사의 본문 내용이 들어가는 컬럼
# 'target': 각 기사가 20개 토픽 중 몇 번째 토픽에 속하는지 나타내는 숫자 라벨(0~19) 컬럼
df = pd.DataFrame({
    'text': news.data,
    'target': news.target
})

# 숫자 라벨(0~19)을 실제 텍스트 형태의 카테고리 이름으로 변환한 'target_name' 컬럼 추가
# 예: 0 -> 'alt.atheism', 1 -> 'comp.graphics', ...
df['target_name'] = df['target'].apply(lambda x: news.target_names[x])

# 생성된 DataFrame의 상위 5개 행 미리보기
print(df.head())

                                                text  target  \
0  From: Mamatha Devineni Ratnam <mr47+@andrew.cm...      10   
1  From: mblawson@midway.ecn.uoknor.edu (Matthew ...       3   
2  From: hilmi-er@dsv.su.se (Hilmi Eren)\nSubject...      17   
3  From: guyd@austin.ibm.com (Guy Dawson)\nSubjec...       3   
4  From: Alexander Samuel McDiarmid <am2o+@andrew...       4   

                target_name  
0          rec.sport.hockey  
1  comp.sys.ibm.pc.hardware  
2     talk.politics.mideast  
3  comp.sys.ibm.pc.hardware  
4     comp.sys.mac.hardware  


In [7]:
# 정규 표현식 처리를 위한 파이썬 표준 라이브러리 불러오기
import re

# scikit-learn 데이터셋 모듈에서 20개 뉴스그룹 데이터를 가져오는 함수 불러오기
from sklearn.datasets import fetch_20newsgroups

# 1. remove 옵션을 지정하지 않고(기본값), 'sci.space' 카테고리의 학습용(train) 원본 데이터를 통째로 로드
raw_news = fetch_20newsgroups(subset='train', categories=['sci.space'])

# 파싱 함수를 테스트하기 위해 가져온 뉴스 데이터 중 첫 번째 문서(문자열) 선택
sample_text = raw_news.data[0]


# 뉴스그룹 텍스트 전체를 받아 Headers, Footers, Quotes, Pure Body의 4개 영역으로 분리하는 함수 정의
def parse_newsgroup_post(text):
    """뉴스그룹 문서에서 Header, Footer, Quote, Pure Body를 각각 분리하여 추출하는 함수"""

    # --- A. Header 추출 ---
    # RFC 규약상 뉴스그룹 헤더와 본문 사이는 첫 번째 빈 줄('\n\n')로 구분됨
    # split("\n\n", 1)을 사용해 가장 첫 번째 빈 줄에서 한 번만 자름
    header_split = text.split("\n\n", 1)
    
    # 첫 번째 빈 줄이 존재하여 자르기가 성공한 경우
    if len(header_split) > 1:
        headers = header_split[0]      # 빈 줄 이전 부분은 Header로 저장
        remainder = header_split[1]    # 빈 줄 이후 부분은 본문 영역(remainder)으로 저장
    # 헤더 구분선이 없는 경우 예외 처리
    else:
        headers = ""
        remainder = text

    # --- B. Footer 추출 ---
    # 표준 서명 구분선인 '\n-- \n' 패턴 뒤에 나오는 모든 텍스트(re.DOTALL)를 탐색
    footer_match = re.search(r"\n-- \n(.*)", remainder, re.DOTALL)
    
    # 서명 구분선 패턴이 발견된 경우
    if footer_match:
        footers = footer_match.group(1)            # 구분선 이후의 모든 텍스트를 Footer로 추출
        remainder = remainder[: footer_match.start()] # Footer 이전 영역으로 본문 영역 축소
    # 서명 구분선이 없는 경우
    else:
        footers = ""

    # --- C. Quotes (인용문) & Pure Body (순수 본문) 추출 ---
    # 인용문과 순수 본문을 라인 단위로 분류하여 담을 리스트 생성
    quotes_list = []
    body_lines = []

    # Footer까지 제거된 남은 본문 영역(remainder)을 한 줄씩 순회하며 검사
    for line in remainder.split("\n"):
        # 줄의 시작 부분의 공백을 제거한 후 '>' 기호로 시작하거나
        # 'In article ... wrote:' 형태로 남의 글을 인용했음을 나타내는 표현 패턴과 일치하는지 확인
        if line.strip().startswith(">") or re.match(
            r"^In article .* wrote:$", line.strip()
        ):
            quotes_list.append(line)  # 인용문 리스트에 해당 줄 추가
        else:
            body_lines.append(line)   # 일반 본문 리스트에 해당 줄 추가

    # 줄 단위로 모인 리스트를 다시 줄바꿈 문자('\n')로 연결하여 하나의 문자열로 재조합
    quotes = "\n".join(quotes_list)
    pure_body = "\n".join(body_lines)

    # 4가지 영역으로 깔끔하게 분리된 결과를 딕셔너리 형태로 반환
    return {
        "headers": headers,
        "footers": footers,
        "quotes": quotes,
        "pure_body": pure_body,
    }


# 첫 번째 샘플 문서를 파싱 함수에 전달하여 실행
parsed_data = parse_newsgroup_post(sample_text)

# 1. 추출된 헤더(Header) 내용 출력
print("=== 1. HEADERS ===")
print(parsed_data["headers"])

# 2. 추출된 푸터(Footer/서명) 내용 출력 (존재하지 않을 경우 대체 문구 출력)
print("\n=== 2. FOOTERS ===")
print(
    parsed_data["footers"]
    if parsed_data["footers"]
    else "(푸터 정보 없음/구분선 미사용)"
)

# 3. 추출된 인용문(Quotes) 내용 출력 (존재하지 않을 경우 대체 문구 출력)
print("\n=== 3. QUOTES ===")
print(
    parsed_data["quotes"]
    if parsed_data["quotes"]
    else "(인용문 없음)"
)

# 4. 추출된 순수 본문(Pure Body) 내용 출력
print("\n=== 4. PURE BODY (순수 본문) ===")
print(parsed_data["pure_body"])

=== 1. HEADERS ===
From: henry@zoo.toronto.edu (Henry Spencer)
Subject: Re: japanese moon landing?
Organization: U of Toronto Zoology
Lines: 21

=== 2. FOOTERS ===
All work is one man's work.             | Henry Spencer @ U of Toronto Zoology
                    - Kipling           |  henry@zoo.toronto.edu  utzoo!henry


=== 3. QUOTES ===
>> there is no such thing as a stable lunar orbit
>
>Is it right??? That is new stuff for me. So it means that  you just can 
>not put a sattellite around around the Moon for too long because its 
>orbit will be unstable??? If so, what is the reason??? Is that because 
>the combined gravitacional atraction of the Sun,Moon and Earth 
>that does not provide a stable  orbit around the Moon???

=== 4. PURE BODY (순수 본문) ===
In article <1qnb9tINN7ff@rave.larc.nasa.gov> C.O.EGALON@LARC.NASA.GOV (CLAUDIO OLIVEIRA EGALON) writes:

Any lunar satellite needs fuel to do regular orbit corrections, and when
its fuel runs out it will crash within months.  The orbits

In [8]:
# scikit-learn 데이터셋 모듈에서 20개 뉴스그룹 데이터를 불러오는 함수 가져오기
from sklearn.datasets import fetch_20newsgroups
# 텍스트 데이터를 단어 빈도수 기반의 수치형 행렬로 변환하는 CountVectorizer 불러오기
from sklearn.feature_extraction.text import CountVectorizer
# 다항 나이브 베이즈(Multinomial Naive Bayes) 분류 모델 불러오기
from sklearn.naive_bayes import MultinomialNB
# 모델의 분류 성능 평가를 위한 정확도(accuracy_score) 함수 가져오기
from sklearn.metrics import accuracy_score

# 1. 사용할 4개 뉴스그룹 카테고리 지정
categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space']

# 2. 학습(Train) 및 검증(Test) 데이터셋 로드
# remove=('headers', 'footers', 'quotes'): 힌트가 되는 정형화된 정보를 제거하여 순수 본문으로만 분류하도록 설정
newsgroups_train = fetch_20newsgroups(subset='train',
                                      remove=('headers', 'footers', 'quotes'),
                                      categories=categories)

newsgroups_test = fetch_20newsgroups(subset='test', 
                                     remove=('headers', 'footers', 'quotes'),
                                     categories=categories)

# 3. 데이터셋 기본 정보 출력
print('#Train set size:', len(newsgroups_train.data))
print('#Test set size:', len(newsgroups_test.data))
print('#Selected categories:', newsgroups_train.target_names)
print('#Train labels:', set(newsgroups_train.target))

# 4. 텍스트 카운트 벡터화(CountVectorizer) 객체 생성
# max_features=2000: 상위 2,000개 단어만 피처로 추출
# min_df=5: 최소 5개 문서 이상 등장한 단어만 포함
# max_df=0.5: 전체 문서의 50%를 초과하는 흔한 단어는 제외
cv = CountVectorizer(max_features=2000, min_df=5, max_df=0.5)

# 5. 텍스트 데이터를 수치형 벡터 행렬로 변환
# 학습 데이터셋(X_train)은 fit_transform()을 사용해 단어장을 학습함과 동시에 변환
X_train_cv = cv.fit_transform(newsgroups_train.data)
y_train = newsgroups_train.target

# 평가 데이터셋(X_test)은 학습 데이터에서 만들어진 단어장을 기준으로 transform()만 수행 (Data Leakage 방지)
X_test_cv = cv.transform(newsgroups_test.data)
y_test = newsgroups_test.target

# 6. 나이브 베이즈 모델 생성 및 학습
nb_model = MultinomialNB()
nb_model.fit(X_train_cv, y_train)

# 7. 검증 데이터셋에 대한 예측 수행 및 성능 평가
pred = nb_model.predict(X_test_cv)
print('#Test Set Accuracy:', round(accuracy_score(y_test, pred), 4))

#Train set size: 2034
#Test set size: 1353
#Selected categories: ['alt.atheism', 'comp.graphics', 'sci.space', 'talk.religion.misc']
#Train labels: {np.int64(0), np.int64(1), np.int64(2), np.int64(3)}
#Test Set Accuracy: 0.7354
